# 从 MLP、LSTM 到 TFT：零售多步需求预测

读取与 Prophet 教程相同的商超、红酒、庭院三套 CSV，输出训练内回看、四个历史滚动起点与最终测试成绩。最后 28 天为 2025-12-04 至 2025-12-31。

从仓库根目录启动 Jupyter，并先安装 `pyproject.toml`，然后 Restart Kernel and Run All。`src/retail_forecast/common.py` 保存共同数据协议，各模型模块保存可检查的完整训练实现。所有输入 CSV 不改写。

训练内回看只描述拟合能力；历史窗口按业务生成机制预先指定，每个起点仅使用当时历史。评价排除闭店/缺货标签，所有方案共用日期和非负/营业日历处理。商超与红酒促销全年重复，因此促销观察窗口仍混合促销和非促销日，另外输出条件切片。


## 1. 数据、业务任务与评估协议

三个案例分别代表平滑加法需求、低销量计数需求与强季节需求。诊断真值 `conditional_mean`、`latent_demand` 不进入特征。

In [ ]:
# 首次使用请在仓库根目录运行：uv sync --group dev
from pathlib import Path
import os,sys
def find_repo_root():
    configured=os.environ.get('RETAIL_FORECAST_ROOT')
    candidates=[Path(configured).expanduser()] if configured else [Path.cwd(),*Path.cwd().parents]
    for candidate in candidates:
        candidate=candidate.resolve()
        if (candidate/'pyproject.toml').is_file() and (candidate/'notebooks').is_dir():return candidate
    raise RuntimeError('找不到仓库根目录；请从仓库内启动 Jupyter，或设置 RETAIL_FORECAST_ROOT')
REPO_ROOT=find_repo_root()
sys.path.insert(0,str(REPO_ROOT/'src'))
import inspect,json
import numpy as np,pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display,Code
from retail_forecast.common import *
display(pd.DataFrame([{'scenario':n,'days':len(f),'cutoff':cutoff,'stockout_days':int(f.stockout.sum())} for n,f in datasets.items()]))
display(pd.DataFrame([{'scenario':n,**w} for n in SCENARIOS for w in windows(n)]))

## 2. 历史窗口与已知未来输入

$$\hat{\mathbf y}_{T+1:T+H}=F_{\boldsymbol\theta}(\mathbf Y_{T-L+1:T},\mathbf X_{T-L+1:T},\mathbf X_{T+1:T+H}).$$

变量：$T$ 为起点，$L=84$ 为历史窗口，$H=28$ 为预测窗口，$\mathbf Y$ 为历史销量，$\mathbf X$ 为可见的日历/业务特征，$F_{\boldsymbol\theta}$ 为参数为 $\boldsymbol\theta$ 的神经网络，帽号表示预测。冒号表示连续索引范围。

MLP 用 84 天销量形成上下文；LSTM 编码历史销量与日历/业务输入；两者将上下文与各未来日已知特征拼接，通过共享输出头直接产生 28 天预测。TFT 使用 PyTorch Forecasting 的完整实现。

本例的 MLP/LSTM 每 7 天构造一个训练窗口，使用掩码 MSE；TFT 沿用上一轮每日滑窗、RMSE 训练协议。因此本例是结构与工程教学，训练预算不完全相同。

**TFT 默认模式**：使用当前权威数据从头训练。已审阅的旧结果保存在 `artifacts/published/`，来自此前通过输入哈希、目标缩放和占位符检查的历史 checkpoint 快照；只有恢复与其 manifest 匹配的数据时，才可显式使用 `checkpoint` 模式。


In [ ]:
scenario='grocery'
X=make_features(datasets[scenario],scenario)
display(X.head());print('Feature columns:',list(X));print('Truth excluded:',not {'y','latent_demand','conditional_mean'}.intersection(X))

## 3. 训练实现与参数检查

完整实现可在配套 Python 模块阅读。下面直接展示当前运行的训练函数，保证解释与代码一致。

In [ ]:
import retail_forecast.neural as neural_models
# 当前唯一数据源默认重新训练；只有恢复 manifest 匹配的历史数据时才使用 checkpoint。
neural_models.TFT_MODE="train"
from retail_forecast.neural import *
display(Code(inspect.getsource(WindowNetwork),language="python"))
display(Code(inspect.getsource(fit_neural),language="python"))

## 4. 实际训练、滚动验证与最终测试

每个历史窗口重新拟合；TFT 当前默认从权威数据重新训练。树和神经网络的内部验证全部位于该起点之前。最终模型的历史回看参数已见过训练标签。Notebook 已清除执行输出，重新 Run All 后的结果写入独立运行目录。

In [ ]:
m,p,details,summary,finals=run_family("neural",NAMES,fit_neural,predict_neural)
display(m.round(4));display(summary.round(4));display(details)

## 5. 预测与性能图

训练、滚动验证和最终测试来自不同时间段。比较 WAPE 时同时考虑销量分母，并查看 MAE、RMSE 和偏差。

In [ ]:
colors=plot_family("neural",NAMES,m,p,summary)

## 6. 参数与诊断

诊断用于发现结构遗漏，未来泛化由时间窗口评价。特征重要性与注意力都需要业务解释，不能直接作为因果效应。

In [ ]:
neural_curves()
display(details[[c for c in ["scenario","model","regime","mode","setup_seconds","recorded_training_seconds","selected_epochs","parameters"] if c in details]])

## 7. 结果口径与系统交付

输出包括预测起点、目标日期、步长、原始预测和营业日历处理后的销售预测。补货系统还要结合提前期、库存和服务水平。当前脚本提供本地可重复实验；生产发布需补齐数据快照、权限、调度、质量门槛与回滚。

In [ ]:
out=RUNS_DIR/'neural'
assert len(m)==3*len(NAMES)*6
assert m.loc[m.split.eq('test'),'n'].nunique()==1
for name,digest in INPUT_HASHES.items():assert hashlib.sha256((DATA_DIR/name).read_bytes()).hexdigest()==digest
print('Exports:',out.resolve())
display(summary.pivot(index='scenario',columns='model',values='WAPE'))